In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import os
from sklearn.model_selection import train_test_split

# Define directories
image_dir = '/kaggle/input/retina-blood-vessel/Data/train/image'
mask_dir = '/kaggle/input/retina-blood-vessel/Data/train/mask'
val_image_dir = '/kaggle/input/retina-blood-vessel/Data/test/image'
val_mask_dir = '/kaggle/input/retina-blood-vessel/Data/test/mask'

# Function to load and preprocess images
def load_image(image_path, mask_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (256, 256))
    image = tf.cast(image, tf.float32) / 255.0
    
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, (256, 256))
    mask = tf.cast(mask, tf.float32) / 255.0
    
    return image, mask

# Create dataset
def create_dataset(image_dir, mask_dir):
    image_files = sorted([os.path.join(image_dir, fname) for fname in os.listdir(image_dir)])
    mask_files = sorted([os.path.join(mask_dir, fname) for fname in os.listdir(mask_dir)])
    
    dataset = tf.data.Dataset.from_tensor_slices((image_files, mask_files))
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

# Create training and validation datasets
train_dataset = create_dataset(image_dir, mask_dir)
val_dataset = create_dataset(val_image_dir, val_mask_dir)

# Batch and prefetch datasets
BATCH_SIZE = 16
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)



In [ ]:
model.summary()

In [ ]:
def additional_encoder(x, filters):
    x = keras.layers.Conv2D(filters, 3, padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Conv2D(filters, 3, padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    return x

In [ ]:
import keras
import keras_cv
import tensorflow as tf # only for data
import tensorflow_io as tfio # for loading .tif files

import cv2
import pandas as pd
import numpy as np
from glob import glob
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt

In [ ]:
segmentation_head = keras.Sequential(
    [
        keras.layers.Conv2D(
            filters=64,  # Increased from 32
            kernel_size=3,  # Changed from 1 to 3
            padding="same",
            use_bias=False,
        ),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.Conv2D(
            filters=32,
            kernel_size=3,
            padding="same",
            use_bias=False,
        ),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.UpSampling2D(size=(4, 4), interpolation="bilinear"),
        keras.layers.Conv2D(
            filters=1,  # Single channel for binary mask
            kernel_size=1,
            use_bias=False,
            padding="same",
            activation="sigmoid",
        ),
    ],
    name="segmentation_head",
)

In [ ]:
class CFG:
    verbose = 1  # Verbosity
    seed = 42  # Random seed
    preset = "deeplab_v3_plus_resnet50_pascalvoc"  
#     image_size = [384, 384]  # Input image size deeplabv3plus
    image_size = [256, 256]  # Input image size
    seg_image_size = [28,28]
    epochs = 25 # Training epochs
    batch_size = 10  # Batch size
    drop_remainder = True  # Drop incomplete batches
    num_classes = 1 # Number of output classes
    cache = True # Save data into memory during training

In [ ]:
import tensorflow as tf
from tensorflow.keras import backend as K

# Previous metric definitions remain the same
def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def iou(y_true, y_pred, smooth=1):
    intersection = K.sum(K.abs(y_true * y_pred), axis=[1,2,3])
    union = K.sum(y_true, [1,2,3])+K.sum(y_pred, [1,2,3])-intersection
    return K.mean((intersection + smooth) / (union + smooth))

def f1_score(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    recall = true_positives / (possible_positives + K.epsilon())
    f1_val = 2*(precision*recall)/(precision+recall+K.epsilon())
    return f1_val

def specificity(y_true, y_pred):
    true_negatives = K.sum(K.round(K.clip((1-y_true) * (1-y_pred), 0, 1)))
    possible_negatives = K.sum(K.round(K.clip(1-y_true, 0, 1)))
    return true_negatives / (possible_negatives + K.epsilon())

def sensitivity(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    return true_positives / (possible_positives + K.epsilon())

# Add AUC metric
auc = tf.keras.metrics.AUC()

# Compile the model with additional metrics including AUC
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', dice_coef, iou, f1_score, specificity, sensitivity, auc]
)
model.summary()

In [ ]:
# Load the full DeepLabV3+ model
backbone = keras_cv.models.DeepLabV3Plus.from_preset(
    CFG.preset,
    input_shape=[*CFG.image_size, 3],
)

# Take only layers from backbone before head
neck_layer_name = backbone.layers[-2].name
out = backbone.get_layer(neck_layer_name).output

# Additional encoder path
additional_features = additional_encoder(backbone.input, 64)
additional_features = keras.layers.MaxPooling2D()(additional_features)
additional_features = additional_encoder(additional_features, 128)
additional_features = keras.layers.MaxPooling2D()(additional_features)
additional_features = additional_encoder(additional_features, 256)

# Combine features
out = keras.layers.Concatenate()([out, additional_features])

# Use newly defined head for segmentation
out = segmentation_head(out)

# Create a new model
model = keras.models.Model(inputs=backbone.input, outputs=out)
# model = unet_model()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()
# Train the model

# OPTIMIZER = keras.optimizers.Adam(learning_rate=1e-4)

# METRICS = [
#     dice_coef,
#     keras.metrics.BinaryAccuracy(name="accuracy"),
#     iou,
# #     sensitivity,
# #     specificity,
# #     auc,
# #     f1_score,
# ]
# # LOSS =  
# model.compile(optimizer=OPTIMIZER, loss=dice_loss, metrics=METRICS)

# # Model Summary
# model.summary()

In [ ]:
history = model.fit(train_dataset, validation_data=val_dataset, epochs=50, callbacks=[
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
])